In [1]:
import os
os.chdir("..")
os.makedirs("monitoring/grafana/provisioning/datasources", exist_ok=True)
os.makedirs("monitoring/grafana/provisioning/dashboards", exist_ok=True)
print(os.getcwd())

/home/s2025ae05941/workspace/heart-mlops


In [2]:
%%writefile monitoring/prometheus.yml
global:
  scrape_interval: 5s

scrape_configs:
  - job_name: heart-api
    metrics_path: /metrics
    static_configs:
      - targets: ["heart-api:8000"]

Writing monitoring/prometheus.yml


In [3]:
%%writefile monitoring/grafana/provisioning/datasources/datasource.yml
apiVersion: 1
datasources:
  - name: Prometheus
    uid: prom
    type: prometheus
    access: proxy
    url: http://prometheus:9090
    isDefault: true

Writing monitoring/grafana/provisioning/datasources/datasource.yml


In [4]:
%%writefile monitoring/grafana/provisioning/dashboards/provider.yml
apiVersion: 1
providers:
  - name: heart
    folder: MLOps
    type: file
    options:
      path: /etc/grafana/provisioning/dashboards

Writing monitoring/grafana/provisioning/dashboards/provider.yml


In [5]:
%%writefile monitoring/grafana/provisioning/dashboards/heart_api.json
{
  "title": "Heart API Monitoring",
  "uid": "heart-api",
  "schemaVersion": 39,
  "refresh": "5s",
  "time": {"from": "now-15m", "to": "now"},
  "panels": [
    {"id": 1, "type": "timeseries", "title": "Requests per second (by endpoint)",
     "gridPos": {"x": 0, "y": 0, "w": 12, "h": 8},
     "datasource": {"type": "prometheus", "uid": "prom"},
     "targets": [{"refId": "A", "legendFormat": "{{endpoint}}",
                  "expr": "sum by (endpoint) (rate(api_requests_total[1m]))"}]},
    {"id": 2, "type": "timeseries", "title": "p95 latency (seconds)",
     "gridPos": {"x": 12, "y": 0, "w": 12, "h": 8},
     "datasource": {"type": "prometheus", "uid": "prom"},
     "targets": [{"refId": "A", "legendFormat": "p95",
                  "expr": "histogram_quantile(0.95, sum by (le) (rate(api_request_latency_seconds_bucket[1m])))"}]},
    {"id": 3, "type": "piechart", "title": "Predictions by label",
     "gridPos": {"x": 0, "y": 8, "w": 12, "h": 8},
     "datasource": {"type": "prometheus", "uid": "prom"},
     "targets": [{"refId": "A", "legendFormat": "{{label}}", "instant": true,
                  "expr": "sum by (label) (predictions_total)"}]},
    {"id": 4, "type": "stat", "title": "Total requests / errors",
     "gridPos": {"x": 12, "y": 8, "w": 12, "h": 8},
     "datasource": {"type": "prometheus", "uid": "prom"},
     "targets": [
       {"refId": "A", "legendFormat": "total", "expr": "sum(api_requests_total)"},
       {"refId": "B", "legendFormat": "errors (4xx/5xx)", "expr": "sum(api_requests_total{status=~\"4..|5..\"}) or vector(0)"}]}
  ]
}

Writing monitoring/grafana/provisioning/dashboards/heart_api.json


In [6]:
%%writefile src/generate_traffic.py
"""Send random patient requests to the API so the monitoring dashboard shows traffic."""
import json
import random
import sys
import time
import urllib.error
import urllib.request

URL = sys.argv[1] if len(sys.argv) > 1 else "http://localhost:8000"
N = int(sys.argv[2]) if len(sys.argv) > 2 else 200


def random_patient():
    return {
        "age": random.randint(30, 77),
        "sex": random.choice([0, 1]),
        "cp": random.choice([1, 2, 3, 4]),
        "trestbps": random.randint(95, 190),
        "chol": random.randint(130, 400),
        "fbs": random.choice([0, 1]),
        "restecg": random.choice([0, 1, 2]),
        "thalach": random.randint(90, 200),
        "exang": random.choice([0, 1]),
        "oldpeak": round(random.uniform(0, 5), 1),
        "slope": random.choice([1, 2, 3]),
        "ca": random.choice([0, 1, 2, 3]),
        "thal": random.choice([3, 6, 7]),
    }


def post(path, payload):
    req = urllib.request.Request(
        URL + path,
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=5) as resp:
        return json.loads(resp.read())


def main():
    counts = {}
    errors = 0
    for i in range(N):
        try:
            if i % 25 == 0:
                post("/predict", {"age": "abc"})  # invalid on purpose -> 422
            result = post("/predict", random_patient())
            counts[result["label"]] = counts.get(result["label"], 0) + 1
        except urllib.error.HTTPError:
            errors += 1
        if i % 10 == 0:
            urllib.request.urlopen(URL + "/health", timeout=5).read()
        time.sleep(0.1)
    print(f"sent {N} predictions to {URL}: {counts}, errors: {errors}")


if __name__ == "__main__":
    main()

Writing src/generate_traffic.py
